In [ ]:
import sys
import os
from dotenv import load_dotenv
import guidance

# Setup local pywhyllm development environment in one line
from notebook_setup import setup_local_pywhyllm
project_root = setup_local_pywhyllm()

# Now import the SimpleModelSuggester
#TFM/pywhyllm/pywhyllm/suggesters --> crear variable 

# project_root = os.path.abspath("/home/moleropa/repositories/master/TFM/pywhyllm/pywhyllm/suggesters/")  # Navigate to TFM/pywhyllm/
# sys.path.insert(0, project_root)

from pywhyllm.suggesters.simple_model_suggester import SimpleModelSuggester

from openai import OpenAI
from portkey_ai import createHeaders
from dotenv import load_dotenv
import time
import base64
from IPython.display import display, Image
from pydantic import BaseModel
import json
load_dotenv()

# # Force reload the module from local source
# import importlib
# import sys

# # Remove the cached module if it exists
# if 'pywhyllm.suggesters.simple_model_suggester' in sys.modules:
#     importlib.reload(sys.modules['pywhyllm.suggesters.simple_model_suggester'])

# from pywhyllm.suggesters.simple_model_suggester import SimpleModelSuggester


In [5]:
azure_model= "gpt-4o-mini" #"GPT-4o-2024-05-13" 
us_base_url = "https://us.aigw.galileo.roche.com/v1"

portkey_headers = createHeaders(config=os.environ["PORTKEY_AZURE_US_CONFIG"])
azure_openai_client = OpenAI(base_url=us_base_url,
            api_key=os.environ["PORTKEY_AZURE_US_API_KEY"],
            default_headers=portkey_headers)


# Guidance con modelo OpenAI + base_url + headers
model = guidance.models.OpenAI(
    #"GPT-4o-2024-05-13",
    azure_model,
    api_key=os.environ["PORTKEY_AZURE_US_API_KEY"],
    base_url=us_base_url,
    default_headers=portkey_headers
)


In [3]:
# Import hallbayes repository for hallucination risk assessment
hallbayes_path = os.path.abspath("../../hallbayes")
sys.path.insert(0, hallbayes_path)

print(f"Hallbayes path added: {hallbayes_path}")

from scripts.hallucination_toolkit import OpenAIBackend, OpenAIItem, OpenAIPlanner

Hallbayes path added: /home/moleropa/repositories/master/TFM/pywhyllm/hallbayes


In [ ]:
# Check environment configuration for hallbayes
print("Checking environment configuration...")
print(f"PORTKEY_AZURE_US_API_KEY exists: {'PORTKEY_AZURE_US_API_KEY' in os.environ}")
print(f"OPENAI_API_KEY exists: {'OPENAI_API_KEY' in os.environ}")

# Hallbayes expects OPENAI_API_KEY, but we're using Azure with Portkey
# Let's set it up for compatibility
if 'PORTKEY_AZURE_US_API_KEY' in os.environ and 'OPENAI_API_KEY' not in os.environ:
    os.environ["OPENAI_API_KEY"] = os.environ["PORTKEY_AZURE_US_API_KEY"]
    print("✅ Set OPENAI_API_KEY from PORTKEY_AZURE_US_API_KEY for hallbayes compatibility")

# Also set the base URL if hallbayes supports it
if 'OPENAI_BASE_URL' not in os.environ:
    os.environ["OPENAI_BASE_URL"] = us_base_url
    print("✅ Set OPENAI_BASE_URL for hallbayes compatibility")

print("Environment configured for hallbayes integration")

Structure: Guidance uses context managers (with guidance.user() and with guidance.assistant()) to structure the conversation
Model State: The lm object accumulates the conversation state as you add to it with +=
Generation: Use guidance.gen() instead of the OpenAI completion parameters
Response Access: Access the generated response via lm['response'] using the name specified in gen()
The guidance approach is more declarative and gives you fine-grained control over the conversation flow, making it particularly useful for complex prompting patterns and multi-step reasoning tasks.

In [22]:
# Make sure to pass the llm parameter explicitly
modeler = SimpleModelSuggester(llm=model)
print("SimpleModelSuggester created successfully with local source code")

SimpleModelSuggester created successfully with local source code


## Test pairwise relationships

In [ ]:
# result = modeler.suggest_pairwise_relationship("ice cream sales", "shark attacks")


In [ ]:
# # doesn't work
# if result[0] is not None:
#     print(f"{result[0]} causes {result[1]}")
# else:
#     print(f"neither causes the other")

## Let's build a graph among our Use case Variables - Diabetes Mellitus 

In [ ]:
# variables = ["ice cream sales", "temperature", "cavities"]
# results = modeler.suggest_relationships(variables)

# results

In [ ]:
variables = ["obesity", "hypertension"]
results = modeler.suggest_relationships(variables)

In [8]:
# Simple timeout handling approach
try:
    print("Starting relationship analysis...")
    start_time = time.time()
    
    variables = ["obesity", "hypertension", "age", "cardiac ischemia", "diabetes mellitus, major acute cardiovascular event"]
    results = modeler.suggest_relationships(variables)
    
    elapsed_time = time.time() - start_time
    print(f"Analysis completed in {elapsed_time:.2f} seconds")
    print("Results:")
    print(results)
    
except (Timeout, ConnectionError) as e:
    print(f"Network timeout or connection error: {e}")
    print("Try running the cell again or check your internet connection")
    
except openai.RateLimitError as e:
    print(f"API rate limit exceeded: {e}")
    print("Please wait a moment and try again")
    
except Exception as e:
    print(f"An error occurred: {type(e).__name__}: {e}")
    print("You may need to check your API configuration or model setup")

Starting relationship analysis...
1/10.0: Querying for relationship between obesity and hypertension


StitchWidget(initial_height='auto', initial_width='100%', srcdoc='<!doctype html>\n<html lang="en">\n<head>\n …

	obesity causes hypertension
2/10.0: Querying for relationship between obesity and age
	age causes obesity
3/10.0: Querying for relationship between obesity and cardiac ischemia
	age causes obesity
3/10.0: Querying for relationship between obesity and cardiac ischemia
	obesity causes cardiac ischemia
4/10.0: Querying for relationship between obesity and diabetes mellitus, major acute cardiovascular event
	obesity causes cardiac ischemia
4/10.0: Querying for relationship between obesity and diabetes mellitus, major acute cardiovascular event
	obesity causes diabetes mellitus, major acute cardiovascular event
5/10.0: Querying for relationship between hypertension and age
	obesity causes diabetes mellitus, major acute cardiovascular event
5/10.0: Querying for relationship between hypertension and age
	age causes hypertension
6/10.0: Querying for relationship between hypertension and cardiac ischemia
	age causes hypertension
6/10.0: Querying for relationship between hypertension and cardia

In [17]:
results.keys()

dict_keys([('obesity', 'hypertension'), ('age', 'obesity'), ('obesity', 'cardiac ischemia'), ('obesity', 'diabetes mellitus, major acute cardiovascular event'), ('age', 'hypertension'), ('hypertension', 'cardiac ischemia'), ('diabetes mellitus, major acute cardiovascular event', 'hypertension'), ('age', 'cardiac ischemia'), ('age', 'diabetes mellitus, major acute cardiovascular event'), ('diabetes mellitus, major acute cardiovascular event', 'cardiac ischemia')])

In [ ]:
results[('obesity', 'hypertension')]
#correct

'The most widely accepted causal relationship between obesity and hypertension is that obesity causes hypertension. This is supported by various studies and medical literature indicating that excess body weight can lead to increased blood pressure due to factors such as increased blood volume, inflammation, and hormonal changes associated with fat tissue. \n\nWhile hypertension can have some correlation with obesity, it is primarily viewed as a consequence rather than a cause of obesity. Additionally, the idea that neither condition causes the other does not align with well-established medical knowledge. \n\nTherefore, the most likely cause-and-effect relationship is:\n\n<answer>A</answer>'

In [ ]:
results[('obesity', 'cardiac ischemia')]
#correct

'To determine which cause-and-effect relationship is more likely between obesity and cardiac ischemia, we can consider the medical evidence and biological mechanisms involved.\n\nA. Obesity causes cardiac ischemia: There is substantial evidence that obesity is a significant risk factor for various cardiovascular diseases, including cardiac ischemia. Obesity can lead to conditions such as hypertension, diabetes, and dyslipidemia, which can, in turn, contribute to the development of cardiac ischemia due to increased strain on the heart and decreased blood flow to cardiac tissue.\n\nB. Cardiac ischemia causes obesity: While cardiac ischemia itself could lead to decreased physical activity due to discomfort or health issues, it does not generally result in obesity directly. In fact, individuals with cardiac issues are often advised to manage their weight for better heart health.\n\nC. Neither obesity nor cardiac ischemia cause each other: This statement might suggest that they are complete

In [ ]:
results[('obesity', 'diabetes mellitus, major acute cardiovascular event')]
#correct

'To evaluate the cause-and-effect relationships presented in options A, B, and C, we need to consider the established medical understanding of obesity and diabetes mellitus in relation to cardiovascular events.\n\n1. **Obesity and Diabetes Mellitus (Option A)**: Obesity is a well-known risk factor for the development of type 2 diabetes mellitus. Excess fat, particularly visceral fat, is associated with insulin resistance, which can lead to diabetes. Additionally, both obesity and diabetes contribute to the risk of major acute cardiovascular events.\n\n2. **Diabetes Mellitus and Obesity (Option B)**: While diabetes can lead to changes in weight and body composition, it is not typically considered a direct cause of obesity. In many cases, obesity precedes and contributes to the development of diabetes, rather than the other way around.\n\n3. **Neither Causes Each Other (Option C)**: While it might seem that obesity and diabetes could be independent conditions, the substantial evidence su

In [ ]:
results[('age', 'hypertension')]
#more or less correct linea discontinuas

"To analyze the cause-and-effect relationships presented in options A, B, and C:\n\nA. Hypertension causes age - This is not a logical relationship, as hypertension is a medical condition that does not influence a person's age. Age is a biological measure that increases with time irrespective of health conditions like hypertension.\n\nB. Age causes hypertension - This is a plausible relationship. As people age, the likelihood of developing hypertension (high blood pressure) increases due to various factors such as changes in blood vessels, increased arterial stiffness, and lifestyle factors that often accumulate over a person's lifespan.\n\nC. Neither hypertension nor age cause each other - While hypertension and age are correlated, stating that neither causes the other neglects the valid observation that advancing age is a significant risk factor for developing hypertension.\n\nGiven this reasoning, the more likely cause-and-effect relationship is that age influences the likelihood of

In [ ]:
results[( ('hypertension',
  'cardiac ischemia'))]
#correct

'To assess the cause-and-effect relationship between hypertension and cardiac ischemia, we should consider the medical understanding of these conditions.\n\nHypertension, or high blood pressure, is a well-established risk factor for various cardiovascular diseases, including cardiac ischemia, which refers to a reduced blood flow (and therefore oxygen) to the heart muscle, often leading to pain or damage.\n\n1. **Hypertension causes cardiac ischemia**: This is consistent with medical literature, as prolonged high blood pressure can lead to changes in the blood vessels and heart, increasing the risk of ischemia due to reduced oxygen supply.\n\n2. **Cardiac ischemia causes hypertension**: While severe cardiac events can lead to stress responses that might temporarily raise blood pressure, ischemia primarily results from underlying conditions, including established hypertension. Therefore, the primary pathway is not typically that ischemia leads to hypertension.\n\n3. **Neither hypertensio

In [ ]:
results[('diabetes mellitus, major acute cardiovascular event',
  'hypertension')]
#wrong

# age cardiac ischemia good

# ('age','diabetes mellitus, major acute cardiovascular event' good

# ('diabetes mellitus, major acute cardiovascular event','cardiac ischemia') good


## Latent confounders

In [4]:
# variables = ["ice cream sales", "temperature", "cavities"]
variables = ["obesity", "hypertension", "age", "cardiac ischemia", "diabetes mellitus", "major acute cardiovascular event"]

latents = modeler.suggest_confounders(variables, "diabetes mellitus", "cardiac ischemia")

print(latents)

StitchWidget(initial_height='auto', initial_width='100%', srcdoc='<!doctype html>\n<html lang="en">\n<head>\n …

['obesity', 'hypertension', 'age']


In [ ]:
# latents = modeler.suggest_confounders(["weight", "diet", "age"], "vitamin c", "cardiovascular health")

# print(latents)

In [4]:
variables = ["obesity", "hypertension", "age", "cardiac ischemia", "diabetes mellitus", "major acute cardiovascular event"]


confounders = modeler.suggest_confounders_custom(variables, "diabetes mellitus", "cardiac ischemia")

print(confounders)

StitchWidget(initial_height='auto', initial_width='100%', srcdoc='<!doctype html>\n<html lang="en">\n<head>\n …

StitchWidget(initial_height='auto', initial_width='100%', srcdoc='<!doctype html>\n<html lang="en">\n<head>\n …

['obesity', 'hypertension', 'age']


it is identifying the confounders correctly if we consider the ground truth our paper Kyriacou

In [6]:
variables = ["obesity", "hypertension", "age", "cardiac ischemia", "diabetes mellitus", "major acute cardiovascular event"]

colliders = modeler.suggest_colliders_custom(variables, "diabetes mellitus", "cardiac ischemia")   
print(colliders)

StitchWidget(initial_height='auto', initial_width='100%', srcdoc='<!doctype html>\n<html lang="en">\n<head>\n …

StitchWidget(initial_height='auto', initial_width='100%', srcdoc='<!doctype html>\n<html lang="en">\n<head>\n …

['hypertension', 'major acute cardiovascular event', 'hypertension', 'major acute cardiovascular event']


## Identification support

### Instrumental variables

In [ ]:
from pywhyllm.suggesters.simple_identification_suggester import SimpleIdentificationSuggester
identifier = SimpleIdentificationSuggester('gpt-4o-mini')

In [ ]:
variables = ["cigarette taxes", "rain", "car sales", "property taxes", "heart attacks"]
ivs = identifier.suggest_iv(variables, "smoking", "birth weight")

ivs

### Backdoor variables

In [ ]:
variables = ["Age", "Sex", "HbA1c", "HDL", "LDL", "eGFR", "Prior MI",
             "Prior Stroke or TIA", "Prior Heart Failure", "Cardiovascular medication",
             "T2DM medication", "Insulin", "Morbid obesity", "First occurrence of Nonfatal myocardial infarction, nonfatal stroke, death from all cause",
             "semaglutide treatment", "Semaglutide medication", "income", "musical taste"]

backdoors = identifier.suggest_backdoor(variables,
                            treatment="semaglutide treatment", outcome = "cardiovascular health")

print(backdoors)

### Frontdoor

In [ ]:
frontdoors = identifier.suggest_frontdoor(variables,
                            treatment="semaglutide treatment", outcome = "cardiovascular health")

print(frontdoors)

## Hallucination Risk Assessment for Causal Discovery

Let's assess the reliability of our causal discovery responses using hallbayes toolkit.

In [ ]:
# First, create the backend normally (this will try to use default OpenAI config)
hallbayes_backend = OpenAIBackend(model=azure_model)

# Then replace its client with the working Azure OpenAI client 
hallbayes_backend.client = azure_openai_client

# Create a test prompt for causal relationship assessment
causal_prompt = """
Based on medical knowledge and the following variables:
- Variable A: Obesity (BMI > 30)
- Variable B: Diabetes Mellitus Type 2

Question: Does obesity cause diabetes mellitus, or does diabetes mellitus cause obesity?
Please provide your answer with medical reasoning.
"""

# Create causal assessment item with reduced parameters for testing
causal_item = OpenAIItem(
    prompt=causal_prompt,
    n_samples=3,  # Reduced for testing
    m=4,          # Reduced for testing
    skeleton_policy="closed_book"
)


In [30]:
planner = OpenAIPlanner(hallbayes_backend, temperature=0.3)

simplify next cell

### Batch Assessment for Multiple Causal Relationships

Now let's assess multiple causal relationships from your diabetes case study.

In [ ]:
# Ejemplo de uso: Validar las relaciones encontradas en tu diabetes case study
variables = ["obesity", "hypertension", "age"]

# 1. Generar relaciones causales
relationships = modeler.suggest_relationships(variables)


In [27]:
relationships

{('obesity',
  'hypertension'): 'To determine which cause-and-effect relationship is more likely between obesity and hypertension, we need to consider the common understanding of these conditions based on medical research.\n\nA. Obesity causes hypertension: This is widely supported in the medical community. Obesity can lead to an increase in blood volume and cardiac output, which can, in turn, elevate blood pressure. Additionally, excess body fat can lead to insulin resistance, inflammation, and changes in hormonal levels that can contribute to hypertension.\n\nB. Hypertension causes obesity: This relationship is less clear. While hypertension can contribute to lifestyle factors that may lead to weight gain, it is not typically seen as a direct cause of obesity. In fact, many individuals with hypertension are not obese.\n\nC. Neither obesity nor hypertension cause each other: While it is true that not all individuals with obesity develop hypertension and vice versa, this statement over

In [43]:
# Unified hallbayes validation with consistency checking
def validate_relationships_with_consistency(relationships, planner, method="closed_book", threshold=0.10, n_runs=3):
    """
    Validates causal relationships using hallbayes with consistency checking
    
    Args:
        relationships: dict from suggest_relationships()
        planner: configured hallbayes planner
        method: "closed_book" or "with_evidence"
        threshold: max acceptable hallucination risk
        n_runs: number of validation runs per relationship (for consistency)
    
    Returns:
        dict: validation results with consistency information
    """
    print(f"Validating {len(relationships)} relationships using {method.upper()} method")
    print(f"Running {n_runs} tests per relationship for consistency check")
    print("=" * 60)
    
    all_results = {}
    
    for rel_idx, ((cause, effect), description) in enumerate(relationships.items(), 1):
        print(f"\n{rel_idx}/{len(relationships)}: {cause} → {effect}")
        
        # Prepare detailed prompt based on method
        if method == "closed_book":
            prompt = f"""
            Medical Knowledge Assessment:
            
            Claim: "{cause}" causes "{effect}"
            
            Question: Based on established medical and scientific knowledge, 
            is this causal relationship scientifically valid?
            
            Answer: Yes/No with brief justification (2-3 established mechanisms).
            """
        elif method == "with_evidence":
            prompt = f"""
            Evidence-Based Assessment:
            
            Provided Evidence: {description}
            
            Claim: "{cause}" causes "{effect}"
            
            Question: Based on the provided evidence and your knowledge, 
            is this causal relationship valid and well-supported?
            
            Answer: Yes/No referencing the provided evidence and additional knowledge.
            """
        else:
            raise ValueError(f"Unknown method: {method}")
        
        # Run multiple validations for consistency
        run_results = []
        valid_count = 0
        
        for run in range(n_runs):
            try:
                item = OpenAIItem(prompt=prompt, n_samples=3, m=4, skeleton_policy=method)
                metrics = planner.run([item], h_star=threshold, isr_threshold=1.0)
                
                if metrics:
                    metric = metrics[0]
                    is_valid = metric.decision_answer
                    risk = metric.roh_bound
                    
                    if is_valid:
                        valid_count += 1
                    
                    run_results.append({
                        'valid': is_valid,
                        'risk': risk,
                        'run': run + 1
                    })
                    
                    status = "VALID" if is_valid else "REJECT"
                    print(f"  Run {run + 1}: {status} (Risk: {risk:.1%})")
                else:
                    run_results.append({'valid': False, 'risk': 1.0, 'run': run + 1, 'error': 'No metrics'})
                    print(f"  Run {run + 1}: ERROR (No metrics)")
                    
            except Exception as e:
                run_results.append({'valid': False, 'risk': 1.0, 'run': run + 1, 'error': str(e)})
                print(f"  Run {run + 1}: ERROR ({str(e)[:50]}...)")
        
        # Calculate consistency and final decision
        consistency_rate = valid_count / n_runs
        avg_risk = sum(r.get('risk', 1.0) for r in run_results) / len(run_results)
        
        # Determine final validation (majority vote)
        final_valid = valid_count > (n_runs / 2)
        
        # Determine consistency level
        if consistency_rate == 1.0 or consistency_rate == 0.0:
            consistency = "CONSISTENT"
        elif consistency_rate >= 0.66:
            consistency = "MOSTLY_CONSISTENT"  
        else:
            consistency = "INCONSISTENT"
        
        all_results[(cause, effect)] = {
            'final_decision': final_valid,
            'consistency': consistency,
            'consistency_rate': consistency_rate,
            'avg_risk': avg_risk,
            'valid_runs': valid_count,
            'total_runs': n_runs,
            'method': method,
            'runs': run_results,
            'description': description
        }
        
        print(f"  → Final: {'VALID' if final_valid else 'REJECT'} | Consistency: {consistency} ({valid_count}/{n_runs})")
    
    return all_results

# Test the unified function with professional prompts
print("Testing unified validation function with professional prompts:")
unified_results = validate_relationships_with_consistency(
    relationships, planner, method="closed_book", threshold=0.10, n_runs=3
)

Testing unified validation function with professional prompts:
Validating 3 relationships using CLOSED_BOOK method
Running 3 tests per relationship for consistency check

1/3: obesity → hypertension
  Run 1: VALID (Risk: 0.0%)
  Run 2: VALID (Risk: 0.0%)
  Run 3: VALID (Risk: 0.0%)
  → Final: VALID | Consistency: CONSISTENT (3/3)

2/3: age → obesity
  Run 1: REJECT (Risk: 51.2%)
  Run 2: REJECT (Risk: 100.0%)
  Run 3: REJECT (Risk: 100.0%)
  → Final: REJECT | Consistency: CONSISTENT (0/3)

3/3: age → hypertension
  Run 1: REJECT (Risk: 76.8%)
  Run 2: VALID (Risk: 0.0%)
  Run 3: REJECT (Risk: 76.8%)
  → Final: REJECT | Consistency: INCONSISTENT (1/3)


In [44]:
unified_results = validate_relationships_with_consistency(
    relationships, planner, method="with_evidence", threshold=0.10, n_runs=3
)

Validating 3 relationships using WITH_EVIDENCE method
Running 3 tests per relationship for consistency check

1/3: obesity → hypertension
  Run 1: VALID (Risk: 0.0%)
  Run 1: VALID (Risk: 0.0%)
  Run 2: VALID (Risk: 0.0%)
  Run 2: VALID (Risk: 0.0%)
  Run 3: VALID (Risk: 0.0%)
  → Final: VALID | Consistency: CONSISTENT (3/3)

2/3: age → obesity
  Run 3: VALID (Risk: 0.0%)
  → Final: VALID | Consistency: CONSISTENT (3/3)

2/3: age → obesity
  Run 1: REJECT (Risk: 100.0%)
  Run 1: REJECT (Risk: 100.0%)
  Run 2: REJECT (Risk: 51.2%)
  Run 2: REJECT (Risk: 51.2%)
  Run 3: REJECT (Risk: 100.0%)
  → Final: REJECT | Consistency: CONSISTENT (0/3)

3/3: age → hypertension
  Run 3: REJECT (Risk: 100.0%)
  → Final: REJECT | Consistency: CONSISTENT (0/3)

3/3: age → hypertension
  Run 1: VALID (Risk: 0.0%)
  Run 1: VALID (Risk: 0.0%)
  Run 2: VALID (Risk: 0.0%)
  Run 2: VALID (Risk: 0.0%)
  Run 3: VALID (Risk: 0.0%)
  → Final: VALID | Consistency: CONSISTENT (3/3)
  Run 3: VALID (Risk: 0.0%)
  → F

In [ ]:
# Summary function for unified results
results = unified_results
def summarize_unified_results(unified_results):
    
    total = len(results)
    valid_relationships = sum(1 for r in results.values() if r['final_decision'])
    consistent_relationships = sum(1 for r in results.values() if r['consistency'] == 'CONSISTENT')
    
    print(f"Total relationships tested: {total}")
    print(f"Valid relationships: {valid_relationships}/{total} ({valid_relationships/total:.1%})")
    print(f"Consistent results: {consistent_relationships}/{total} ({consistent_relationships/total:.1%})")
    
    print(f"\nDetailed Results:")
    print("-" * 40)
    
    for (cause, effect), result in results.items():
        decision = "VALID" if result['final_decision'] else "REJECT"
        consistency = result['consistency']
        rate = result['consistency_rate']
        avg_risk = result['avg_risk']
        
        print(f"{cause} → {effect}:")
        print(f"  Decision: {decision} | Consistency: {consistency} ({rate:.1%}) | Avg Risk: {avg_risk:.1%}")
    
    print(f"\nRecommendation:")
    print("-" * 20)
    high_confidence = sum(1 for r in results.values() 
                         if r['final_decision'] and r['consistency'] == 'CONSISTENT')
    
    print(f"Use {high_confidence} relationships with VALID + CONSISTENT results for further analysis.")
    if high_confidence < total:
        uncertain = total - high_confidence
        print(f"Review {uncertain} relationships that are inconsistent or rejected.")

# Show summary of unified results
summarize_unified_results(results)


VALIDATION SUMMARY
Total relationships tested: 3
Valid relationships: 2/3 (66.7%)
Consistent results: 3/3 (100.0%)

Detailed Results:
----------------------------------------
obesity → hypertension:
  Decision: VALID | Consistency: CONSISTENT (100.0%) | Avg Risk: 0.0%
age → obesity:
  Decision: REJECT | Consistency: CONSISTENT (0.0%) | Avg Risk: 83.7%
age → hypertension:
  Decision: VALID | Consistency: CONSISTENT (100.0%) | Avg Risk: 0.0%

Recommendation:
--------------------
Use 2 relationships with VALID + CONSISTENT results for further analysis.
Review 1 relationships that are inconsistent or rejected.
